In [ ]:
from pathlib import Path
import random

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from ultralytics import YOLO

import pandas as pd

In [ ]:
normal_root = Path("normal_slide_path")
aml_root = Path("AML_slide_path")
model_path = "yolo_model_weights.pt"

num_regions_list = [1, 2, 5, 10, 20, 50]
num_bootstrap_iters = 5

blast_class_id = 1
normal_class_id = 2

# clinically used thresholds
normal_threshold = 0.05   # predict normal if blast_ratio < this
aml_threshold = 0.20     # predict AML if blast_ratio > this

random_seed = 42

In [ ]:
yolo_model = YOLO(model_path)
print(yolo_model.names)

In [ ]:
def get_slide_folders(root_folder):
    return sorted(
        [p for p in root_folder.iterdir() if p.is_dir()],
        key=lambda p: int(p.name)
    )

NL_slides = get_slide_folders(normal_root)
AML_slides = get_slide_folders(aml_root)

print("Normal slides:", [p.name for p in NL_slides])
print("AML slides:", [p.name for p in AML_slides])
print("Total slides:", len(NL_slides) + len(AML_slides))

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def get_region_files(slide_folder):
    region_folder = slide_folder / "focus_regions" / "high_mag_unannotated"
    if not region_folder.exists():
        return []
    return sorted([
        p for p in region_folder.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ])

In [ ]:
def compute_slide_blast_ratio(slide_folder, num_regions, rng):
    """
    Randomly sample num_regions images from a slide, run YOLO,
    and return:
        blast_count, normal_count, blast_ratio
    """
    image_files = get_region_files(slide_folder)

    if len(image_files) == 0:
        return 0, 0, 0.0

    n_to_sample = min(num_regions, len(image_files))
    sampled_files = rng.sample(image_files, n_to_sample)

    blast_total = 0
    normal_total = 0

    for img_path in sampled_files:
        pred_results = yolo_model(str(img_path), verbose=False)

        for res in pred_results:
            if res.boxes is None or res.boxes.cls is None:
                continue

            ids = res.boxes.cls.cpu().numpy().astype(int)
            blast_total += int((ids == blast_class_id).sum())
            normal_total += int((ids == normal_class_id).sum())

    denom = blast_total + normal_total
    blast_ratio = blast_total / denom if denom > 0 else 0.0

    return blast_total, normal_total, blast_ratio

In [ ]:
def compute_accuracy_and_auc(num_regions, iteration_idx):
    """
    For one bootstrap iteration at a given region count:
    - sample regions per slide
    - compute blast ratios
    - compute accuracy and AUC
    """
    rng = random.Random(random_seed + iteration_idx * 1000 + num_regions)

    NL_correct = 0
    AML_correct = 0

    y_true = []
    y_pred = []

    # normal slides
    for slide_folder in NL_slides:
        blast_count, normal_count, blast_ratio = compute_slide_blast_ratio(
            slide_folder, num_regions, rng
        )

        y_true.append(0)
        y_pred.append(blast_ratio)

        if blast_ratio <= normal_threshold:
            NL_correct += 1

    # AML slides
    for slide_folder in AML_slides:
        blast_count, normal_count, blast_ratio = compute_slide_blast_ratio(
            slide_folder, num_regions, rng
        )

        y_true.append(1)
        y_pred.append(blast_ratio)

        if blast_ratio >= aml_threshold:
            AML_correct += 1

    total_slides = len(NL_slides) + len(AML_slides)
    accuracy = (NL_correct + AML_correct) / total_slides
    auc = roc_auc_score(y_true, y_pred)

    return accuracy, auc, NL_correct, AML_correct

In [ ]:
accuracy_boot = []
auc_boot = []

for num_regions in num_regions_list:
    acc_row = []
    auc_row = []

    print(f"Sampling {num_regions} regions per slide")

    for iter_idx in range(num_bootstrap_iters):
        accuracy, auc, NL_correct, AML_correct = compute_accuracy_and_auc(
            num_regions=num_regions,
            iteration_idx=iter_idx
        )

        acc_row.append(accuracy)
        auc_row.append(auc)

        print(
            f"Iter {iter_idx + 1}: "
            f"NL correct = {NL_correct}/{len(NL_slides)}, "
            f"AML correct = {AML_correct}/{len(AML_slides)}, "
            f"Accuracy = {accuracy:.4f}, AUC = {auc:.4f}"
        )

    accuracy_boot.append(acc_row)
    auc_boot.append(auc_row)

accuracy_boot = np.array(accuracy_boot)
auc_boot = np.array(auc_boot)

print("\nAccuracy bootstrap array:")
print(accuracy_boot)

print("\nAUC bootstrap array:")
print(auc_boot)

In [ ]:
save_folder = Path("Figure_4D_bootstrap_results")
save_folder.mkdir(exist_ok=True)

accuracy_boot_array = np.array(accuracy_boot)
auc_boot_array = np.array(auc_boot)

np.save(save_folder / "accuracy_boot.npy", accuracy_boot_array)
np.save(save_folder / "auc_boot.npy", auc_boot_array)

pd.DataFrame(accuracy_boot_array).to_csv(save_folder / "accuracy_boot.csv", index=False)
pd.DataFrame(auc_boot_array).to_csv(save_folder / "auc_boot.csv", index=False)

print("Saved files to:", save_folder.resolve())

In [ ]:
acc_mean = accuracy_boot.mean(axis=1) * 100
acc_std  = accuracy_boot.std(axis=1)  * 100

auc_mean = auc_boot.mean(axis=1) * 100
auc_std  = auc_boot.std(axis=1)  * 100

print("Accuracy mean (%):", acc_mean)
print("Accuracy std (%):", acc_std)
print("AUC mean (%):", auc_mean)
print("AUC std (%):", auc_std)

In [ ]:
plt.rcParams["font.family"] = "Arial"

n_images = np.array(num_regions_list)

acc_mean = accuracy_boot.mean(axis=1) * 100
acc_std  = accuracy_boot.std(axis=1) * 100
auc_mean = auc_boot.mean(axis=1) * 100
auc_std  = auc_boot.std(axis=1) * 100

fig, ax = plt.subplots(figsize=(6.8, 4.7), dpi=150)

# bootstrap points
for xi, (acc_row, auc_row) in enumerate(zip(accuracy_boot, auc_boot)):
    x = n_images[xi]
    jitter = (np.random.rand(len(acc_row)) - 0.5) * 0.18

    ax.scatter(
        np.full(len(acc_row), x) + jitter,
        np.array(acc_row) * 100,
        s=50,
        color="#E74C3C",
        alpha=0.5,
        edgecolors="none",
        zorder=2
    )

    ax.scatter(
        np.full(len(auc_row), x) + jitter,
        np.array(auc_row) * 100,
        s=50,
        color="#2E86C1",
        alpha=0.5,
        marker="s",
        edgecolors="none",
        zorder=2
    )

# mean ± std curves
ax.errorbar(
    n_images,
    acc_mean,
    yerr=acc_std,
    fmt="o-",
    color="#E74C3C",
    ecolor="#E74C3C",
    elinewidth=1.0,
    capsize=3,
    capthick=1.0,
    markersize=7,
    linewidth=0.9,
    label="Accuracy",
    zorder=3
)

ax.errorbar(
    n_images,
    auc_mean,
    yerr=auc_std,
    fmt="s--",
    color="#2E86C1",
    ecolor="#2E86C1",
    elinewidth=1.0,
    capsize=3,
    capthick=1.0,
    markersize=7,
    linewidth=0.9,
    label="AUC",
    zorder=3
)

ax.set_xscale("log")
ax.set_xticks(n_images)
ax.set_xticklabels([str(x) for x in n_images], fontsize=14)
ax.set_ylim(-3, 105)

ax.set_xlabel("Regions Per Slide", fontsize=26, fontweight="bold")
ax.set_ylabel("Score (%)", fontsize=26, fontweight="bold")

ax.tick_params(axis="y", labelsize=18, length=2, width=0.6)
ax.tick_params(axis="x", labelsize=18, length=2, width=0.6)

ax.grid(False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(0.7)
ax.spines["bottom"].set_linewidth(0.7)

ax.legend(
    loc="lower left",
    frameon=False,
    fontsize=16,
    handlelength=1.6,
    markerscale=0.9
)

plt.tight_layout()
plt.show()

In [ ]:
save_name = "bootstrap_accuracy_auc.png"
fig.savefig(save_name, dpi=300, bbox_inches="tight")
print(f"Saved figure to {save_name}")